# Kahn's Algorithm: Build Order Simulator

Kahn's algorithm performs a topological sort: it arranges tasks so every prerequisite comes before the task that needs it.

Topological sorting is the quiet logic behind build systems, package managers, course prerequisites, spreadsheet recalculation, and workflow engines. Whenever work has arrows of dependency, this is the shape hiding underneath.

In this notebook, you will build it with small objects: tasks, dependencies, a task graph, and a runner that unlocks tasks in order.

<details>
<summary>Big idea</summary>

Start with tasks that have no prerequisites. Each time you finish one, remove its outgoing dependency arrows. Any task with no remaining prerequisites becomes ready.

</details>

## 1. The Mental Model

A dependency graph is a directed graph:

- **Task**: something to do, like `Bake Cake`
- **Dependency**: an arrow from prerequisite to later task
- **In-degree**: how many prerequisites a task still has
- **Ready queue**: tasks with in-degree `0`
- **Schedule**: the sorted order you build

Kahn's algorithm repeats one move: take a ready task, add it to the schedule, then update the tasks it unlocks.

<details>
<summary>Hint: why does this work?</summary>

A task with in-degree `0` has no unfinished prerequisites, so it is safe to schedule next.

</details>

## 2. Build the Objects

Implementation plan:

1. `Task` stores a task name.
2. `TaskGraph` stores prerequisite arrows.
3. `SortStep` records snapshots so the algorithm feels like a simulation.
4. `KahnRunner` owns the topological sort and cycle detection.

<details>
<summary>Implementation hint</summary>

Track an `in_degree` score for each task. When a score reaches `0`, that task joins the ready queue.

</details>

In [ ]:
from dataclasses import dataclass, field

### Define a Task

- 

In [ ]:


@dataclass(frozen=True, order=True)
class Task:
    name: str

    def __str__(self) -> str:
        return self.name

    def __format__(self, spec: str) -> str:
        return format(self.name, spec)

### Define the Task Graph

- 

In [ ]:
@dataclass
class TaskGraph:
    next_tasks: dict[Task, set[Task]] = field(default_factory=dict)
    previous_tasks: dict[Task, set[Task]] = field(default_factory=dict)

    def add_task(self, task: Task) -> None:
        self.next_tasks.setdefault(task, set())
        self.previous_tasks.setdefault(task, set())

    def require(self, prerequisite: Task, task: Task) -> None:
        self.add_task(prerequisite)
        self.add_task(task)
        self.next_tasks[prerequisite].add(task)
        self.previous_tasks[task].add(prerequisite)

    def tasks(self) -> list[Task]:
        return sorted(self.next_tasks)

    def unlocked_by(self, task: Task) -> list[Task]:
        return sorted(self.next_tasks.get(task, set()))

    def prerequisites_for(self, task: Task) -> list[Task]:
        return sorted(self.previous_tasks.get(task, set()))

    def describe(self) -> str:
        rows = []
        for task in self.tasks():
            prerequisites = self.prerequisites_for(task)
            label = ", ".join(str(item) for item in prerequisites) or "none"
            rows.append(f"{task:>14} needs {label}")
        return "\n".join(rows)


### Define the sort step

- 

In [ ]:
@dataclass
class SortStep:
    current: Task | None
    action: str
    ready: list[Task]
    schedule: list[Task]
    in_degree: dict[Task, int]

### Define the Sort Result

In [ ]:
@dataclass
class SortResult:
    schedule: list[Task]
    cycle_tasks: list[Task]
    steps: list[SortStep]

### Define Khan's Algorithm

- 

In [ ]:
class KahnRunner:
    def __init__(self, graph: TaskGraph):
        self.graph = graph

    def sort(self) -> SortResult:
        in_degree = {task: len(self.graph.prerequisites_for(task)) for task in self.graph.tasks()}
        ready = sorted(task for task, count in in_degree.items() if count == 0)
        schedule: list[Task] = []
        steps = [self._snapshot(None, "start with tasks that have no prerequisites", ready, schedule, in_degree)]

        while ready:
            current = ready.pop(0)
            schedule.append(current)
            steps.append(self._snapshot(current, f"schedule {current}", ready, schedule, in_degree))

            for unlocked_task in self.graph.unlocked_by(current):
                in_degree[unlocked_task] -= 1
                if in_degree[unlocked_task] == 0:
                    ready.append(unlocked_task)
                    ready.sort()
                    action = f"{unlocked_task} is now ready"
                else:
                    action = f"{unlocked_task} still needs {in_degree[unlocked_task]} prerequisite(s)"

                steps.append(self._snapshot(current, action, ready, schedule, in_degree))

        cycle_tasks = [task for task in self.graph.tasks() if task not in schedule]
        if cycle_tasks:
            steps.append(self._snapshot(None, "cycle found: no remaining task can become ready", ready, schedule, in_degree))

        return SortResult(schedule=schedule, cycle_tasks=cycle_tasks, steps=steps)

    def _snapshot(
        self,
        current: Task | None,
        action: str,
        ready: list[Task],
        schedule: list[Task],
        in_degree: dict[Task, int],
    ) -> SortStep:
        return SortStep(
            current=current,
            action=action,
            ready=ready.copy(),
            schedule=schedule.copy(),
            in_degree=in_degree.copy(),
        )

## 3. Create a Tiny Project

Now make a project where some tasks must happen before others.

Read `A needs B` as: `B` must happen before `A`.

<details>
<summary>Hint: which tasks are ready first?</summary>

Tasks with no prerequisites are ready at the start. In this graph, `Sketch Idea` should be available immediately.

</details>

In [2]:
sketch = Task("Sketch Idea")
shop = Task("Buy Supplies")
cut = Task("Cut Pieces")
paint = Task("Paint Pieces")
assemble = Task("Assemble Model")
pack = Task("Pack Display")

project = TaskGraph()
project.require(sketch, shop)
project.require(shop, cut)
project.require(shop, paint)
project.require(cut, assemble)
project.require(paint, assemble)
project.require(assemble, pack)

print(project.describe())

Assemble Model needs Cut Pieces, Paint Pieces
  Buy Supplies needs Sketch Idea
    Cut Pieces needs Buy Supplies
  Pack Display needs Assemble Model
  Paint Pieces needs Buy Supplies
   Sketch Idea needs none


## 4. Run Kahn's Algorithm

The runner returns a `SortResult` with three things:

- `schedule`: a valid topological order
- `cycle_tasks`: tasks that could not be scheduled
- `steps`: snapshots for replaying the process

<details>
<summary>Quick check</summary>

`Assemble Model` should appear after both `Cut Pieces` and `Paint Pieces`.

</details>

In [3]:
runner = KahnRunner(project)
result = runner.sort()

print("Build order:")
for number, task in enumerate(result.schedule, start=1):
    print(f"{number}. {task}")

print("\nCycle found:", bool(result.cycle_tasks))

Build order:
1. Sketch Idea
2. Buy Supplies
3. Cut Pieces
4. Paint Pieces
5. Assemble Model
6. Pack Display

Cycle found: False


## 5. Replay the Sort

The replay shows the ready queue, the schedule so far, and the remaining prerequisite count for each task.

<details>
<summary>Hint: what does in-degree mean here?</summary>

A task's in-degree is the number of prerequisite arrows still pointing into it. When that number hits `0`, the task is ready.

</details>

**Trace model.** Define `SortReplay`, the structure used to capture replayable algorithm state.


In [ ]:
class SortReplay:
    def __init__(self, steps: list[SortStep]):
        self.steps = steps

    def show(self, limit: int | None = None) -> None:
        selected_steps = self.steps if limit is None else self.steps[:limit]

        for number, step in enumerate(selected_steps, start=1):
            current = step.current or "none"
            print(f"Step {number}: {step.action}")
            print("  current :", current)
            print("  ready   :", self._format_tasks(step.ready))
            print("  schedule:", self._format_tasks(step.schedule))
            print("  in-degree:", self._format_in_degree(step.in_degree))
            print()

    def _format_tasks(self, tasks: list[Task]) -> str:
        return " -> ".join(str(task) for task in tasks) or "empty"

    def _format_in_degree(self, in_degree: dict[Task, int]) -> str:
        return ", ".join(f"{task}={count}" for task, count in sorted(in_degree.items()))

replay = SortReplay(result.steps)


**Inspect the result.** Evaluate the expression and read the output before changing parameters.


In [ ]:
replay.show(limit=9)


## 6. Your Experiments

Try changing one thing at a time:

- Add a new task before `Pack Display`
- Add another starting task with no prerequisites
- Remove one dependency and compare the schedule
- Create a cycle and watch Kahn's algorithm catch it

<details>
<summary>Challenge</summary>

Predict which task enters the ready queue next. Then run the replay and compare.

</details>

In [5]:
cycle_graph = TaskGraph()
plan = Task("Plan Menu")
shop_food = Task("Shop Food")
cook = Task("Cook Meal")
serve = Task("Serve Meal")

cycle_graph.require(plan, shop_food)
cycle_graph.require(shop_food, cook)
cycle_graph.require(cook, serve)
cycle_graph.require(serve, shop_food)

cycle_result = KahnRunner(cycle_graph).sort()

print("Scheduled tasks:", " -> ".join(str(task) for task in cycle_result.schedule) or "none")
print("Blocked by cycle:", ", ".join(str(task) for task in cycle_result.cycle_tasks) or "none")

Scheduled tasks: Plan Menu
Blocked by cycle: Cook Meal, Serve Meal, Shop Food


## Visual Trace + Rigor Studio

**Problem frame.** Order tasks while respecting directed dependencies.

**Interactive animation target.** Animate in-degree counts, ready nodes, emitted order, and cycle detection.

**Correctness handle.** Every emitted node has no unmet incoming dependency.

**Complexity handle.** O(V + E) with adjacency lists and an in-degree queue.

**Failure mode to test.** A directed cycle means no valid topological order exists.

**Studio task.** Add one dependency that creates a cycle and show exactly where the ready queue empties.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
